# Google ADK Tutorial — Part 1: Installation & Setup

## What You'll Learn
- What Google ADK is and why it exists
- How to install and configure ADK
- The architecture of an ADK application
- Run your first agent (programmatically, not via CLI)

> **Target version:** ADK 2.3.x (June 2026)


## 1.1 What is Google ADK?

**ADK (Agent Development Kit)** is Google's open-source framework for
building AI agents. An *agent* is an LLM-powered program that can use
tools (web search, code execution, APIs), maintain conversation state,
and collaborate with other agents.

### Why ADK instead of raw Gemini API?

| Raw Gemini API | ADK |
|---|---|
| You manage conversation history manually | Automatic session management |
| Tool calling requires manual loop | Built-in tool execution loop |
| One model, one call | Multi-agent orchestration |
| No state persistence | Session state + memory service |
| No streaming infrastructure | Streaming events out of the box |
| No deployment story | `adk deploy cloud_run` |

### Core Architecture

```
┌─────────────────────────────────────────────────┐
│                    Runner                        │
│  ┌─────────┐   ┌──────────┐   ┌─────────────┐  │
│  │  Agent   │──▶│  Session  │◀──│  Memory Svc  │  │
│  │ (LLM +   │   │ (state,   │   │ (cross-      │  │
│  │  tools)  │   │  history) │   │  session)    │  │
│  └─────────┘   └──────────┘   └─────────────┘  │
└─────────────────────────────────────────────────┘
```

- **Agent**: The "brain" — an LLM with instructions and tools
- **Runner**: The execution engine that runs the agent loop
- **Session**: Stores conversation history within a session (a session = a chat)
- **Memory Service**: Stores facts across sessions (user preferences, etc.)


### Models and Providers

ADK is **model-agnostic**. You can use:
- **Gemini** (default) — requires `GOOGLE_API_KEY`
- **Any LiteLLM model** — OpenAI, Anthropic, OpenRouter, Ollama, etc.

Set the model when creating an agent:
```python
Agent(model='gemini-2.5-flash')           # Google Gemini (default)
Agent(model='gpt-4o')                     # OpenAI
Agent(model='openrouter:deepseek/deepseek-v3')  # OpenRouter → 200+ models
Agent(model='ollama/llama3.2')            # Local Ollama
```

### The Agent Loop (How ADK Works Internally)

When you call `runner.run()`, ADK executes this loop:

```
1. Send user message + conversation history to LLM
2. LLM responds with either:
   a) Text → return to user, DONE
   b) Tool call → execute the tool, feed result back to LLM, GOTO 1
3. Repeat until LLM produces text (not a tool call)
```

This is the **ReAct (Reasoning + Acting) pattern** — the agent reasons
about what to do, acts via tools, and reasons about the results.


In [8]:
# ─── Verify Python version ───────────────────────────────────────
# ADK 2.x requires Python 3.11+ (uses modern typing features)
import sys
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 11), "ADK 2.x needs Python 3.11+"
print("✅ Python version OK")


Python version: 3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]
✅ Python version OK


In [23]:
# ─── Verify ADK is installed ─────────────────────────────────────
# The importlib.metadata module reads package versions from pip
from importlib.metadata import version
try:
    adk_ver = version("google-adk")  # Query installed version
    print(f"✅ ADK installed: v{adk_ver}")
except Exception:
    print('❌ ADK not found. Run: pip install "google-adk[extensions,eval]"')


✅ ADK installed: v2.3.0


## 1.2 API Key Configuration

### Where to Get a Key
1. Go to https://aistudio.google.com/app/apikey
2. Click "Create API Key"
3. Copy the key (starts with `AIza...`)

### How ADK Finds Your Key
ADK's Gemini client looks for the key in this order:
1. `GOOGLE_API_KEY` environment variable
2. Application Default Credentials (for Vertex AI)

**Best practice**: Store the key in a `.env` file (never commit to git!)
and load it at runtime. This cell does exactly that.


In [24]:
# ─── Load API key from .env file ───────────────────────────────────
# Google ADK needs GOOGLE_API_KEY to call Gemini models.
# The .env file (in this directory) stores your key safely.
# This cell reads it and sets the environment variable for the session.
import os
from dotenv import load_dotenv, find_dotenv

# Load the .env file
load_dotenv(find_dotenv())
has_key = bool(os.environ.get("GOOGLE_API_KEY", ""))
print(f'API key: {"✅ LOADED" if has_key else "❌ NOT FOUND — create .env file"}')


API key: ✅ LOADED


## 1.3 Core ADK Imports — What Each Module Does

| Import | Purpose |
|---|---|
| `google.adk.Agent` | The agent class — defines the LLM, instructions, and tools |
| `google.adk.runners.Runner` | Executes the agent loop (send→LLM→respond→repeat) |
| `google.adk.sessions.InMemorySessionService` | Stores conversation state in RAM (use DB-backed for production) |
| `google.genai.types.Content, Part` | Message types — **NOT** from `google.adk.types` in v2.x! |
| `asyncio` | Session creation is async in ADK 2.x; we need `await` |

> **Important ADK 2.x change:** Content and Part live in `google.genai.types`,
> not `google.adk.types`. This is because ADK v2.x unified its type system
> with the Google GenAI SDK.


In [13]:
# ─── Core ADK imports for every project ────────────────────────────
# ADK 2.x uses a clean import structure. Here are the four essentials:
import asyncio  # Required: session creation is async in ADK 2.x
from google.adk import Agent  # The core agent class — your building block
from google.adk.runners import Runner  # Executes agents and manages sessions
from google.adk.sessions.in_memory_session_service import InMemorySessionService  # Stores conversation state
from google.genai.types import Content, Part  # Message types (NOTE: google.genai.types, NOT google.adk.types!)
print("✅ Core imports ready")


✅ Core imports ready


## 1.4 Your First Agent — A Line-by-Line Walkthrough

### What We're Building
A minimal agent that greets the user. This is the "Hello World" of ADK.

### The 4 Steps to Run Any Agent
1. **Define the Agent** — name, model, instruction (system prompt)
2. **Create a Session** — a container for conversation state (async!)
3. **Create a Runner** — the execution engine
4. **Call `runner.run()`** — sends message, iterates over streaming events

### Understanding Events
`runner.run()` returns a **generator of Events**. Each Event represents
one "thing that happened":
- A text chunk from the LLM (streaming)
- A tool call being made
- A tool result being received
- An error

You iterate over events to stream the response in real time.


In [29]:
# ═══════════════════════════════════════════════════════════════════
# YOUR FIRST AGENT — Step-by-step with inline comments
# ═══════════════════════════════════════════════════════════════════

# ─── Step 1: Define the Agent ────────────────────────────────────
# An Agent is the core building block. It wraps an LLM with a system
# prompt (instruction) and optional tools.
from google.adk.models import Gemini  # Import Gemini model wrapper

async def main():
    agent = Agent(
        name="greeter",               # Unique ID for this agent (used in logs/routing)
        # model="gemini-2.5-flash",     # Which LLM to use — fast and capable
        model=Gemini(
            model="gemini-2.5-flash", 
            api_key="YOUR_GOOGLE_API_KEY"),
        instruction="You are a friendly greeter. Answer in ONE sentence. 😊",
        # ↑ The instruction is the "system prompt" — it defines the agent's personality
        #   and behavior. Be specific! Vague instructions = unpredictable behavior.
    )

    # ─── Step 2: Create a Session (ASYNC in ADK 2.x!) ──────────
    # A session is like a "chat instance" — it stores the conversation
    # history and state for one user's interaction.
    # InMemorySessionService keeps everything in RAM (good for dev).
    # For production, use a database-backed session service.
    svc = InMemorySessionService()
    session = await svc.create_session(
        app_name="tutorial",    # Groups sessions under an app
        user_id="user1",        # Identifies the user
        session_id="s1"         # Unique session ID (like a chat thread ID)
    )
    print(f"Session created: {session.id}")  # Verify session exists

    # ─── Step 3: Create the Runner ──────────────────────────────
    # The Runner ties together: Agent + Session Service + Memory Service
    # It orchestrates the agent loop (send→LLM→respond→repeat)
    runner = Runner(
        agent=agent,               # Which agent to run
        app_name="tutorial",       # Must match session's app_name
        session_service=svc,        # Where sessions are stored
    )

    # ─── Step 4: Build the Message ──────────────────────────────
    # ADK uses Content and Part from google.genai.types
    # Content = a message (like one line in a chat)
    # Part = a piece of content inside a message (text, image, function call, etc.)
    msg = Content(
        role="user",  # Who sent this message? "user" or "model"
        parts=[Part.from_text(text="Hello! Who are you?")]
    )

    # ─── Step 5: Run and Stream Events ──────────────────────────
    # runner.run() returns a GENERATOR of events — iterate to stream
    # Each event has: .author (who produced it), .content (what was produced)
    print("User: Hello! Who are you?\n")
    for event in runner.run(
        user_id="user1",            # Must match session's user_id
        session_id=session.id,      # Which session to continue
        new_message=msg             # The new user message
    ):
        # ─── Filter: only show model text responses ────────────
        if event.content and event.content.parts:
            for part in event.content.parts:
                # part.text = plain text from the LLM
                # part.thought = "thinking" content (skip these)
                # part.function_call = LLM wants to call a tool
                # part.function_response = tool result
                # if part.text and not part.thought:
                    print(f"[{event.author}]: {part.text}")
    print("\n✅ First agent ran successfully!")

# ─── Execute the async function ─────────────────────────────────
# In Jupyter, use "await" (not asyncio.run) because Jupyter already
# has a running event loop.
await main()


Session created: s1
User: Hello! Who are you?

[greeter]: Hello! I'm your friendly greeter, here to welcome you.

✅ First agent ran successfully!


### What Just Happened? (The Agent Loop, Step by Step)

1. **Runner received** the user message `"Hello! Who are you?"`
2. **Runner sent to LLM**: The message + agent instruction + conversation history
3. **LLM generated**: A text response (no tool calls needed — simple greeting)
4. **Runner yielded the Event** containing the text
5. **Our loop printed it**

The agent had no tools, so the LLM could only produce text. In later
notebooks, we'll add tools so the agent can search the web, run code,
and call APIs.

### Why Async Sessions?

ADK 2.x made session creation async because production session services
(Cloud Firestore, Spanner) require async I/O. The in-memory service is
also async for API consistency — you write the same code in dev and prod.


## 1.5 The `adk` CLI (Reference)

ADK ships with a CLI for project management. You don't need it for
programmatic usage (which is what these notebooks teach), but it's
useful for scaffolding and deployment:

| Command | What it does |
|---|---|
| `adk create my_agent` | Scaffolds a new agent project with agent.py, .env, __init__.py |
| `adk run my_agent` | Runs agent in interactive terminal mode |
| `adk web my_agent --port 8001` | Launches a browser-based chat UI at http://localhost:8001 |
| `adk deploy cloud_run` | Deploys to Google Cloud Run (one command!) |
| `adk deploy agent_engine` | Deploys to Vertex AI Agent Engine (enterprise) |

## Summary

| Concept | Key point |
|---|---|
| **Agent** | LLM + instruction + tools. The building block. |
| **Runner** | Executes the agent loop (send→LLM→respond→repeat) |
| **Session** | Stores conversation state for one chat (async creation) |
| **Event** | One thing that happened (text, tool call, error) |
| **Content/Part** | Message format — from `google.genai.types`, NOT `google.adk.types` |

**Next:** [02_first_agent.ipynb](./02_first_agent.ipynb) — deep dive into Agent, Runner, Sessions.
